# PhysioGraphSleep — Training Pipeline

Single-channel EEG sleep staging with heterogeneous graph neural networks.

**Target**: Sleep-EDF-20 — 20 subjects, 5-class (W/N1/N2/N3/REM)

---

**Kullanım:**
1. Colab'da Runtime → Change runtime type → **G4 GPU**
2. Hücreleri sırayla çalıştır (repo GitHub'dan otomatik çekilir)

## 1. Setup & Clone Repo

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Channels contain different.*")
warnings.filterwarnings("ignore", message=".*Highpass cutoff frequency.*")
warnings.filterwarnings("ignore", message=".*Lowpass cutoff frequency.*")
os.environ["MNE_LOGGING_LEVEL"] = "ERROR"

# Expand /dev/shm for multi-worker DataLoader on Colab
!df -h /dev/shm
!sudo mount -o remount,size=2G /dev/shm
!df -h /dev/shm

In [ ]:
REPO_URL = "https://github.com/0nur0duncu/physiographsleep.git"
PROJECT_DIR = "/content/physiographsleep"

if os.path.exists(PROJECT_DIR):
    # Guncelle
    !cd {PROJECT_DIR} && git pull
    print(f"Repo guncellendi: {PROJECT_DIR}")
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    print(f"Repo klonlandi: {PROJECT_DIR}")

In [ ]:
os.chdir("/content")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
print(f"Working dir: {os.getcwd()}")
print(f"Icerik: {os.listdir(PROJECT_DIR)}")

In [ ]:
!pip install -q -r {PROJECT_DIR}/requirements.txt

In [ ]:
import torch
import numpy as np
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Configuration

In [ ]:
from physiographsleep.configs import ExperimentConfig
from physiographsleep.utils.reproducibility import set_seed, get_device

config = ExperimentConfig()

# === Colab path overrides ===
config.data.data_dir = "physiographsleep/dataset/sleep-edfx"
config.data.cache_dir = "physiographsleep/dataset/cache"
config.data.num_subjects = 20
config.data.train_subjects = 14
config.data.val_subjects = 3
config.data.test_subjects = 3
config.data.batch_size = 64
config.data.num_workers = 4

# v2 Curriculum — Stage B/C extended for better N1 convergence
config.train.curriculum.stage_a_epochs = 30
config.train.curriculum.stage_b_epochs = 25
config.train.curriculum.stage_c_epochs = 25
config.train.patience = 10

# Checkpoints (Colab local)
config.train.checkpoint_dir = "checkpoints"
config.train.log_dir = "logs"

set_seed(config.seed)
device = get_device("auto")
print(f"Device: {device}")
print(f"Subjects: {config.data.num_subjects} (train={config.data.train_subjects}, val={config.data.val_subjects}, test={config.data.test_subjects})")
print(f"Batch size: {config.data.batch_size}")
print(f"Curriculum: A={config.train.curriculum.stage_a_epochs}, B={config.train.curriculum.stage_b_epochs}, C={config.train.curriculum.stage_c_epochs}")

## 3. Dataset Download & Loading

In [ ]:
from physiographsleep.data.loader import load_sleep_edf

data = load_sleep_edf(config.data)

for split_name, split_data in data.items():
    epochs = split_data["epochs"]
    labels = split_data["labels"]
    spectral = split_data.get("spectral")
    print(f"{split_name:5s}: {len(labels):6d} epochs | shape={epochs.shape} | spectral={spectral.shape if spectral is not None else 'N/A'}")

In [ ]:
# Class distribution
STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]
train_labels = data["train"]["labels"]
counts = np.bincount(train_labels, minlength=5)
total = len(train_labels)
print("Train class distribution:")
for i, name in enumerate(STAGE_NAMES):
    print(f"  {name}: {counts[i]:6d} ({100*counts[i]/total:.1f}%)")

## 4. DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from physiographsleep.data.dataset import SleepEDFDataset
from physiographsleep.data.transforms import SleepTransforms

train_transform = SleepTransforms(config.data)

train_ds = SleepEDFDataset(
    data["train"]["epochs"], data["train"]["labels"],
    config=config.data, transform=train_transform,
    spectral=data["train"].get("spectral"),
)
val_ds = SleepEDFDataset(
    data["val"]["epochs"], data["val"]["labels"],
    config=config.data,
    spectral=data["val"].get("spectral"),
)

val_loader = DataLoader(
    val_ds, batch_size=config.data.batch_size,
    shuffle=False, num_workers=config.data.num_workers,
    pin_memory=True,
    persistent_workers=config.data.num_workers > 0,
    prefetch_factor=2 if config.data.num_workers > 0 else None,
)

print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")

## 5. Model

In [ ]:
from physiographsleep.models.physiographsleep import PhysioGraphSleep

model = PhysioGraphSleep(config.model)
param_counts = model.count_parameters()

print("Parameter counts:")
for name, count in param_counts.items():
    print(f"  {name}: {count:,}")

In [ ]:
# Sanity check - forward pass
batch = next(iter(val_loader))
with torch.no_grad():
    model.eval()
    sig = batch["signal"][:2]
    spec = batch["spectral"][:2] if "spectral" in batch else torch.zeros(2, config.data.seq_len, 5, 42)
    out = model(sig, spec)
    print(f"Input:  signal={sig.shape}")
    print(f"Output: stage={out['stage'].shape}, boundary={out['boundary'].shape}")
    print("Forward pass OK")

## 6. Training

In [ ]:
# Free GPU memory from any previous runs
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, {torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

In [ ]:
from physiographsleep.models.losses import MultiTaskLoss
from physiographsleep.data.spectral import SpectralFeatureExtractor
from physiographsleep.training.trainer import Trainer
from physiographsleep.utils.logging_utils import setup_logger

# Class-balanced inverse-frequency weights (no manual boost)
class_counts = np.bincount(data["train"]["labels"], minlength=5)
class_weights = 1.0 / (class_counts.astype(np.float32) + 1e-6)
class_weights = class_weights / class_weights.sum() * 5

class_weights_tensor = torch.from_numpy(class_weights).float()
print(f"Class weights: {[f'{w:.3f}' for w in class_weights_tensor.tolist()]}")

loss_fn = MultiTaskLoss(
    config.train.loss,
    class_weights=class_weights_tensor,
)
spectral_ext = SpectralFeatureExtractor(config.data)
logger = setup_logger(log_dir=config.train.log_dir)

trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    train_dataset=train_ds,
    train_labels=data["train"]["labels"],
    val_loader=val_loader,
    config=config.train,
    data_config=config.data,
    device=device,
    spectral_extractor=spectral_ext,
)

print("Trainer ready.")

In [ ]:
# Auto-resume: detect existing stage checkpoints
from physiographsleep.utils.io_utils import load_checkpoint

ckpt_dir = config.train.checkpoint_dir
start_stage = "A"

for stage_name, stage_file, next_stage in [
    ("C", "stage-c.pt", None),
    ("B", "stage-b.pt", "C"),
    ("A", "stage-a.pt", "B"),
]:
    ckpt_path = os.path.join(ckpt_dir, stage_file)
    if os.path.exists(ckpt_path):
        ckpt = load_checkpoint(ckpt_path, device)
        model.load_state_dict(ckpt["model"])
        mf1 = ckpt["metrics"]["macro_f1"]
        print(f"Found {stage_file} (MF1={mf1:.4f})")
        if next_stage:
            start_stage = next_stage
            print(f"Resuming from Stage {next_stage}")
        else:
            start_stage = None
            best_metrics = ckpt["metrics"]
            print("All stages complete — skip to post-processing")
        break
else:
    print("No checkpoint found, starting from Stage A")

if start_stage:
    best_metrics = trainer.train(start_stage=start_stage)

    print("\n" + "="*50)
    print("TRAINING COMPLETE")
    print("="*50)
    from physiographsleep.evaluation.metrics import MetricsCalculator
    print(MetricsCalculator.format_report(best_metrics))
else:
    from physiographsleep.evaluation.metrics import MetricsCalculator
    print("\nBest model metrics:")
    print(MetricsCalculator.format_report(best_metrics))

## 7. Test Evaluation

In [ ]:
from physiographsleep.training.evaluator import Evaluator
from physiographsleep.utils.io_utils import load_checkpoint
from physiographsleep.evaluation.metrics import MetricsCalculator

# Load best Stage C checkpoint
ckpt_path = os.path.join(config.train.checkpoint_dir, "stage-c.pt")
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    print(f"Stage C checkpoint loaded (val MF1={ckpt['metrics']['macro_f1']:.4f})")
else:
    print("No stage-c.pt found, using last model state")

# Evaluate on test set with logits
if "test" in data:
    test_ds = SleepEDFDataset(
        data["test"]["epochs"], data["test"]["labels"],
        config=config.data,
        spectral=data["test"].get("spectral"),
    )
    test_loader = DataLoader(
        test_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )

    evaluator = Evaluator(device)
    test_metrics, test_logits, test_labels = evaluator.evaluate(
        model, test_loader, spectral_ext, return_logits=True,
    )

    print("\nTest Results (raw argmax):")
    print(MetricsCalculator.format_report(test_metrics))
else:
    print("No test split available")

## 8. HMM Post-processing

In [ ]:
from physiographsleep.evaluation.postprocessing import HMMPostProcessor, LogitBiasOptimizer

if "test" in data:
    # --- Step 1: Optimize logit biases on VALIDATION set ---
    val_ds_eval = SleepEDFDataset(
        data["val"]["epochs"], data["val"]["labels"],
        config=config.data,
        spectral=data["val"].get("spectral"),
    )
    val_loader_eval = DataLoader(
        val_ds_eval, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )

    _, val_logits, val_labels = evaluator.evaluate(
        model, val_loader_eval, spectral_ext, return_logits=True,
    )

    print("=" * 50)
    print("LOGIT BIAS OPTIMIZATION (on validation set)")
    print("=" * 50)
    bias_opt = LogitBiasOptimizer(num_classes=5)
    bias_opt.fit(val_logits, val_labels)

    # --- Step 2: Apply to test set ---
    all_preds_raw = test_logits.argmax(axis=1)
    all_preds_biased = bias_opt.apply(test_logits)

    # --- Step 3: HMM on top of biased predictions ---
    hmm = HMMPostProcessor(num_classes=5)
    hmm.fit(data["train"]["labels"])
    all_preds_hmm = hmm.decode(all_preds_biased)

    calc = MetricsCalculator()
    raw_m = calc.compute_all(test_labels, all_preds_raw)
    biased_m = calc.compute_all(test_labels, all_preds_biased)
    hmm_m = calc.compute_all(test_labels, all_preds_hmm)

    print("\n" + "=" * 50)
    print("TEST RESULTS COMPARISON")
    print("=" * 50)
    print("\n1) Raw argmax:")
    print(MetricsCalculator.format_report(raw_m))
    print(f"\n2) + Logit bias (MF1 {raw_m['macro_f1']:.4f} → {biased_m['macro_f1']:.4f}, {biased_m['macro_f1']-raw_m['macro_f1']:+.4f}):")
    print(MetricsCalculator.format_report(biased_m))
    print(f"\n3) + Logit bias + HMM (MF1 {biased_m['macro_f1']:.4f} → {hmm_m['macro_f1']:.4f}, {hmm_m['macro_f1']-biased_m['macro_f1']:+.4f}):")
    print(MetricsCalculator.format_report(hmm_m))

    # Save biases for inference
    all_preds = all_preds_hmm  # best pipeline
    all_labels = test_labels

## 9. Confusion Matrix

In [ ]:
from physiographsleep.evaluation.visualization import plot_confusion_matrix

if "test" in data:
    save_dir = "outputs"
    os.makedirs(save_dir, exist_ok=True)

    cm_raw = MetricsCalculator.confusion_matrix(test_labels, all_preds_raw)
    cm_biased = MetricsCalculator.confusion_matrix(test_labels, all_preds_biased)
    cm_hmm = MetricsCalculator.confusion_matrix(test_labels, all_preds_hmm)

    plot_confusion_matrix(
        cm_raw,
        save_path=os.path.join(save_dir, "cm_raw.png"),
        title=f"Raw argmax (MF1={raw_m['macro_f1']:.3f})",
    )
    plot_confusion_matrix(
        cm_biased,
        save_path=os.path.join(save_dir, "cm_biased.png"),
        title=f"+ Logit bias (MF1={biased_m['macro_f1']:.3f})",
    )
    plot_confusion_matrix(
        cm_hmm,
        save_path=os.path.join(save_dir, "cm_hmm.png"),
        title=f"+ Bias + HMM (MF1={hmm_m['macro_f1']:.3f})",
    )
    print(f"Confusion matrices saved to {save_dir}")

## 10. Save Final Model

In [ ]:
save_dir = "outputs"
os.makedirs(save_dir, exist_ok=True)

final_path = os.path.join(save_dir, "final_model.pt")
torch.save({
    "model": model.state_dict(),
    "config": {
        "num_subjects": config.data.num_subjects,
        "batch_size": config.data.batch_size,
        "seq_len": config.data.seq_len,
        "curriculum": f"A={config.train.curriculum.stage_a_epochs}/B={config.train.curriculum.stage_b_epochs}/C={config.train.curriculum.stage_c_epochs}",
    },
    "metrics": best_metrics,
    "param_counts": param_counts,
}, final_path)

print(f"Saved: {final_path}")
print(f"Params: {param_counts['total']:,}")
print(f"Best MF1: {best_metrics.get('macro_f1', 'N/A')}")

# Colab'dan indirmek icin
from google.colab import files
files.download(final_path)